# Fine-tune PaddleOCR tiếng Việt

Pipeline duy nhất: tải dữ liệu, kiểm tra, baseline, chạy 5 cấu hình, chọn cấu hình tốt nhất, fine-tune cuối, test và phân tích lỗi.

## Cài đặt

In [13]:
# !python -m pip install -q gdown pyyaml rapidfuzz
# RTX 50-series (Blackwell, sm_120) needs a nightly PaddlePaddle build; stable 2.x has no sm_120 kernels
# !python -m pip install --pre -q paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/nightly/cu126/
# !pip show paddlepaddle-gpu
# !nvidia-smi

In [15]:
from pathlib import Path
import copy
import json
import glob
import math
import os
import pickle
import random
import shutil
import subprocess
import sys
import time
import unicodedata
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import yaml
from rapidfuzz.distance import Levenshtein
from IPython.display import display

# pip-installed nvidia-* wheels (cudnn, cublas, ...) aren't on the loader path by default
nvidia_lib_dirs = sorted(glob.glob(str(Path(sys.exec_prefix) / "lib/python*/site-packages/nvidia/*/lib")))
os.environ["LD_LIBRARY_PATH"] = ":".join(nvidia_lib_dirs + [os.environ.get("LD_LIBRARY_PATH", "")])

SEED = 2026
SEARCH_EPOCHS = 1
FINAL_EPOCHS = 4
BATCH_SIZE = 16
IGNORE_SPACE = True

DRIVE_ID = "1_NKW1CL49NKtnT92ddaNZwGcaFJOkUgM"
PADDLEOCR_COMMIT = "2661c7c0ef5c613e8f93c6e93b2e052399f0f854"

ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
WORK = ROOT / "paddleocr_vi"
PADDLE_DIR = WORK / "PaddleOCR"
DATA_EXTRACT = WORK / "data"
CONFIG_DIR = WORK / "configs"
OUTPUT_DIR = WORK / "output"
RESULTS_DIR = WORK / "results"
WEIGHT_DIR = WORK / "weights"
LOG_DIR = WORK / "logs"

for p in [WORK, DATA_EXTRACT, CONFIG_DIR, OUTPUT_DIR, RESULTS_DIR, WEIGHT_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)


In [16]:
if not PADDLE_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/PaddlePaddle/PaddleOCR.git", str(PADDLE_DIR)], check=True)

subprocess.run(["git", "-C", str(PADDLE_DIR), "fetch", "--all"], check=True)
subprocess.run(["git", "-C", str(PADDLE_DIR), "checkout", PADDLEOCR_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PADDLE_DIR / "requirements.txt")], check=True)

import paddle
print("Paddle:", paddle.__version__)
print("PaddleOCR commit:", subprocess.check_output(["git", "-C", str(PADDLE_DIR), "rev-parse", "HEAD"], text=True).strip())
print("Seed:", SEED)

HEAD is now at 2661c7c0ef Update README (#18272)


Paddle: 2.6.2
PaddleOCR commit: 2661c7c0ef5c613e8f93c6e93b2e052399f0f854
Seed: 2026



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## Dữ liệu

In [17]:
import gdown

ZIP_PATH = WORK / "vi_rec_100k.zip"
if not ZIP_PATH.exists():
    gdown.download(id=DRIVE_ID, output=str(ZIP_PATH), quiet=False)

if not any(DATA_EXTRACT.iterdir()):
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(DATA_EXTRACT)

# the downloaded archive nests a second zip (e.g. DACK/data/vi_rec_100k.zip) with the real labels/images
for nested_zip in DATA_EXTRACT.rglob("*.zip"):
    if "__MACOSX" in nested_zip.parts or nested_zip.name.startswith("._"):
        continue
    marker = nested_zip.parent / ".extracted"
    if marker.exists():
        continue
    with zipfile.ZipFile(nested_zip, "r") as zf:
        zf.extractall(nested_zip.parent)
    marker.touch()


def find_one(root, name):
    matches = [m for m in root.rglob(name) if "__MACOSX" not in m.parts]
    if not matches:
        raise FileNotFoundError(name)
    return matches[0]

TRAIN_FILE = find_one(DATA_EXTRACT, "rec_train.txt")
VAL_FILE = find_one(DATA_EXTRACT, "rec_val.txt")
TEST_FILE = find_one(DATA_EXTRACT, "rec_test.txt")
DICT_FILE = find_one(DATA_EXTRACT, "vi_dict.txt")
sample_line = next(x for x in TRAIN_FILE.read_text(encoding="utf-8").splitlines() if x.strip())
sample_rel = sample_line.split("\t", 1)[0]
candidates = [TRAIN_FILE.parent, TRAIN_FILE.parent.parent, DATA_EXTRACT]
DATA_DIR = next((x for x in candidates if (x / sample_rel).exists()), TRAIN_FILE.parent)

print("Data:", DATA_DIR)
print("Train:", TRAIN_FILE)
print("Val:", VAL_FILE)
print("Test:", TEST_FILE)
print("Dict:", DICT_FILE)

Data: /workspace/SinoNom-NLP/paddleocr_vi/data/DACK/data
Train: /workspace/SinoNom-NLP/paddleocr_vi/data/DACK/data/rec_train.txt
Val: /workspace/SinoNom-NLP/paddleocr_vi/data/DACK/data/rec_val.txt
Test: /workspace/SinoNom-NLP/paddleocr_vi/data/DACK/data/rec_test.txt
Dict: /workspace/SinoNom-NLP/paddleocr_vi/data/DACK/data/vi_dict.txt


## Kiểm tra dữ liệu

In [18]:
def canonical_key(rel):
    p = Path(rel)
    if p.is_absolute():
        return os.path.normpath(os.path.relpath(p, DATA_DIR))
    return os.path.normpath(rel)


def read_labels(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n\r")
            if not line:
                continue
            rel, text = line.split("\t", 1)
            rows.append((canonical_key(rel), unicodedata.normalize("NFC", text)))
    return rows

train_rows = read_labels(TRAIN_FILE)
val_rows = read_labels(VAL_FILE)
test_rows = read_labels(TEST_FILE)

lengths = np.array([len(t) for _, t in train_rows])
MAX_TEXT_LENGTH = max(25, int(math.ceil(lengths.max() / 8) * 8))

stats = pd.DataFrame({
    "split": ["train", "val", "test"],
    "so_mau": [len(train_rows), len(val_rows), len(test_rows)]
})
display(stats)
print("Do dai p50/p90/p95/p99/max:", np.percentile(lengths, [50, 90, 95, 99]).tolist(), int(lengths.max()))
print("max_text_length:", MAX_TEXT_LENGTH)

with open(DICT_FILE, "r", encoding="utf-8") as f:
    dict_chars = {unicodedata.normalize("NFC", x.rstrip("\n\r")) for x in f if x.rstrip("\n\r")}

all_chars = set("".join(t for _, t in train_rows + val_rows + test_rows)) - {" "}
missing_chars = sorted(all_chars - dict_chars)
missing_images = [rel for rel, _ in train_rows + val_rows + test_rows if not (DATA_DIR / rel).exists()]

print("Ky tu ngoai vi_dict:", missing_chars[:50], "count=", len(missing_chars))
print("Anh khong tim thay:", len(missing_images))
if missing_chars:
    raise ValueError("vi_dict.txt khong phu ky tu trong nhan")
if missing_images:
    raise FileNotFoundError(missing_images[:10])

,split,so_mau
0,train,93997
1,val,3000
2,test,3003


Do dai p50/p90/p95/p99/max: [57.0, 68.0, 73.0, 79.0] 80
max_text_length: 80
Ky tu ngoai vi_dict: [] count= 0
Anh khong tim thay: 0


In [19]:
GENERIC_URL = "https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_mobile_rec_pretrained.pdparams"
LATIN_URL = "https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/latin_PP-OCRv5_mobile_rec_pretrained.pdparams"
GENERIC_WEIGHT = WEIGHT_DIR / "PP-OCRv5_mobile_rec_pretrained.pdparams"
LATIN_WEIGHT = WEIGHT_DIR / "latin_PP-OCRv5_mobile_rec_pretrained.pdparams"


def download(url, path):
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return path


download(GENERIC_URL, GENERIC_WEIGHT)
download(LATIN_URL, LATIN_WEIGHT)
print(GENERIC_WEIGHT)
print(LATIN_WEIGHT)

/workspace/SinoNom-NLP/paddleocr_vi/weights/PP-OCRv5_mobile_rec_pretrained.pdparams
/workspace/SinoNom-NLP/paddleocr_vi/weights/latin_PP-OCRv5_mobile_rec_pretrained.pdparams


In [20]:
BASE_CONFIG = PADDLE_DIR / "configs/rec/PP-OCRv5/PP-OCRv5_mobile_rec.yml"


def run_process(args, log_name):
    log_path = LOG_DIR / log_name
    start = time.time()
    with open(log_path, "w", encoding="utf-8") as log:
        p = subprocess.run(args, cwd=PADDLE_DIR, stdout=log, stderr=subprocess.STDOUT, text=True)
    if p.returncode != 0:
        tail = log_path.read_text(encoding="utf-8", errors="ignore").splitlines()[-40:]
        print("\n".join(tail))
        raise RuntimeError("Command failed: " + " ".join(map(str, args)))
    return time.time() - start


def run_infer(config_path, label_file, result_txt, log_name, pretrained=None, checkpoint=None):
    args = [
        sys.executable, "tools/infer_rec.py", "-c", str(config_path), "-o",
        f"Global.infer_img={DATA_DIR}",
        f"Global.infer_list={label_file}",
        f"Global.save_res_path={result_txt}",
    ]
    if pretrained is not None:
        args.append(f"Global.pretrained_model={pretrained}")
    if checkpoint is not None:
        args.append(f"Global.checkpoints={checkpoint}")
    return run_process(args, log_name)


def read_prediction_file(path):
    preds = {}
    scores = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 3:
                continue
            full_path, pred, score = parts[0], parts[1], parts[-1]
            rel = os.path.normpath(os.path.relpath(full_path, DATA_DIR))
            preds[rel] = unicodedata.normalize("NFC", pred)
            try:
                scores[rel] = float(score)
            except ValueError:
                scores[rel] = None
    return preds, scores


def metric_text(text):
    text = unicodedata.normalize("NFC", text)
    return text.replace(" ", "") if IGNORE_SPACE else text


def evaluate_predictions(label_file, pred_txt, jsonl_path=None):
    gt_rows = read_labels(label_file)
    preds, scores = read_prediction_file(pred_txt)
    correct = 0
    distances = []
    records = []
    for rel, gt in gt_rows:
        pred = preds.get(rel, "")
        a, b = metric_text(pred), metric_text(gt)
        correct += int(a == b)
        distances.append(Levenshtein.normalized_distance(a, b))
        records.append({"image": rel, "gt": gt, "pred": pred})
    result = {
        "acc": correct / len(gt_rows),
        "norm_edit_dis": 1 - float(np.mean(distances)),
        "n": len(gt_rows),
        "missing_pred": sum(1 for rel, _ in gt_rows if rel not in preds),
    }
    if jsonl_path is not None:
        with open(jsonl_path, "w", encoding="utf-8") as f:
            for r in records:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return result, records

## Baseline

In [21]:
BASELINE_TXT = RESULTS_DIR / "baseline_test.txt"
BASELINE_JSONL = RESULTS_DIR / "pred_test_baseline.jsonl"

baseline_time = run_infer(
    BASE_CONFIG,
    TEST_FILE,
    BASELINE_TXT,
    "baseline_test.log",
    pretrained=GENERIC_WEIGHT,
)
baseline_metric, baseline_records = evaluate_predictions(TEST_FILE, BASELINE_TXT, BASELINE_JSONL)
baseline_metric["inference_seconds"] = baseline_time
pd.DataFrame([baseline_metric])

[2026/08/19 16:45:02] ppocr INFO:             MultiLabelEncode : 
[2026/08/19 16:45:02] ppocr INFO:                 gtc_encode : NRTRLabelEncode
[2026/08/19 16:45:02] ppocr INFO:             KeepKeys : 
[2026/08/19 16:45:02] ppocr INFO:                 keep_keys : ['image', 'label_ctc', 'label_gtc', 'length', 'valid_ratio']
[2026/08/19 16:45:02] ppocr INFO:     loader : 
[2026/08/19 16:45:02] ppocr INFO:         batch_size_per_card : 128
[2026/08/19 16:45:02] ppocr INFO:         drop_last : True
[2026/08/19 16:45:02] ppocr INFO:         num_workers : 8
[2026/08/19 16:45:02] ppocr INFO:         shuffle : True
[2026/08/19 16:45:02] ppocr INFO:     sampler : 
[2026/08/19 16:45:02] ppocr INFO:         divided_factor : [8, 16]
[2026/08/19 16:45:02] ppocr INFO:         first_bs : 128
[2026/08/19 16:45:02] ppocr INFO:         fix_bs : False
[2026/08/19 16:45:02] ppocr INFO:         is_training : True
[2026/08/19 16:45:02] ppocr INFO:         name : MultiScaleSampler
[2026/08/19 16:45:02] ppoc

RuntimeError: Command failed: /workspace/.venv/bin/python tools/infer_rec.py -c /workspace/SinoNom-NLP/paddleocr_vi/PaddleOCR/configs/rec/PP-OCRv5/PP-OCRv5_mobile_rec.yml -o Global.infer_img=/workspace/SinoNom-NLP/paddleocr_vi/data/DACK/data Global.infer_list=/workspace/SinoNom-NLP/paddleocr_vi/data/DACK/data/rec_test.txt Global.save_res_path=/workspace/SinoNom-NLP/paddleocr_vi/results/baseline_test.txt Global.pretrained_model=/workspace/SinoNom-NLP/paddleocr_vi/weights/PP-OCRv5_mobile_rec_pretrained.pdparams

## 5 cấu hình

Mỗi cấu hình chạy 1 epoch để chọn nhanh trên validation. Cấu hình tốt nhất sẽ được train lại 4 epoch trên toàn bộ train.

In [ ]:
CONFIGS = [
    {"name": "c1_generic_640_aug", "init": "generic", "width": 640, "recconaug": True},
    {"name": "c2_generic_960_aug", "init": "generic", "width": 960, "recconaug": True},
    {"name": "c3_latin_640_aug", "init": "latin", "width": 640, "recconaug": True},
    {"name": "c4_latin_960_aug", "init": "latin", "width": 960, "recconaug": True},
    {"name": "c5_latin_960_noaug", "init": "latin", "width": 960, "recconaug": False},
]
pd.DataFrame(CONFIGS)

In [ ]:
def make_config(spec, epochs, suffix="search"):
    with open(BASE_CONFIG, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    name = f"{spec['name']}_{suffix}"
    width = int(spec["width"])
    pretrained = GENERIC_WEIGHT if spec["init"] == "generic" else LATIN_WEIGHT
    save_dir = OUTPUT_DIR / name

    g = cfg["Global"]
    g["model_name"] = name
    g["epoch_num"] = int(epochs)
    g["seed"] = SEED
    g["distributed"] = False
    g["save_model_dir"] = str(save_dir)
    g["save_epoch_step"] = 1
    g["eval_batch_step"] = [0, 1000000]
    g["pretrained_model"] = str(pretrained)
    g["checkpoints"] = None
    g["character_dict_path"] = str(DICT_FILE)
    g["max_text_length"] = MAX_TEXT_LENGTH
    g["use_space_char"] = True
    g["d2s_train_image_shape"] = [3, 48, width]

    cfg["Optimizer"]["lr"]["learning_rate"] = 0.0005
    cfg["Optimizer"]["lr"]["warmup_epoch"] = 0 if epochs <= 1 else 1
    cfg["Metric"]["ignore_space"] = IGNORE_SPACE

    head_list = cfg["Architecture"]["Head"]["head_list"]
    for head in head_list:
        if "NRTRHead" in head:
            head["NRTRHead"]["max_text_length"] = MAX_TEXT_LENGTH

    train_ds = cfg["Train"]["dataset"]
    train_ds["data_dir"] = str(DATA_DIR)
    train_ds["label_file_list"] = [str(TRAIN_FILE)]

    new_transforms = []
    for op in train_ds["transforms"]:
        key = next(iter(op))
        if key == "RecConAug":
            if not spec["recconaug"]:
                continue
            op[key]["image_shape"] = [48, width, 3]
            op[key]["max_text_length"] = MAX_TEXT_LENGTH
        if key == "MultiLabelEncode":
            if op[key] is None:
                op[key] = {}
            op[key]["max_text_length"] = MAX_TEXT_LENGTH
        new_transforms.append(op)
    train_ds["transforms"] = new_transforms

    sampler = cfg["Train"]["sampler"]
    sampler["scales"] = [[width, 32], [width, 48], [width, 64]]
    sampler["first_bs"] = BATCH_SIZE
    sampler["fix_bs"] = True
    cfg["Train"]["loader"]["batch_size_per_card"] = BATCH_SIZE
    cfg["Train"]["loader"]["num_workers"] = 4

    eval_ds = cfg["Eval"]["dataset"]
    eval_ds["data_dir"] = str(DATA_DIR)
    eval_ds["label_file_list"] = [str(VAL_FILE)]
    for op in eval_ds["transforms"]:
        key = next(iter(op))
        if key == "RecResizeImg":
            op[key]["image_shape"] = [3, 48, width]
        if key == "MultiLabelEncode":
            if op[key] is None:
                op[key] = {}
            op[key]["max_text_length"] = MAX_TEXT_LENGTH
    cfg["Eval"]["loader"]["batch_size_per_card"] = BATCH_SIZE
    cfg["Eval"]["loader"]["num_workers"] = 4

    path = CONFIG_DIR / f"{name}.yml"
    with open(path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)
    return path, save_dir


search_config_paths = {}
for spec in CONFIGS:
    path, save_dir = make_config(spec, SEARCH_EPOCHS, "search")
    search_config_paths[spec["name"]] = (path, save_dir)

print("Configs:", CONFIG_DIR)

In [ ]:
def checkpoint_epoch(prefix):
    states = Path(str(prefix) + ".states")
    if not states.exists():
        return 0
    with open(states, "rb") as f:
        data = pickle.load(f)
    return int(data.get("epoch", 0))


def train_with_resume(cfg_path, save_dir, epochs, log_name):
    latest = save_dir / "latest"
    time_file = save_dir / "train_seconds.txt"
    total_seconds = float(time_file.read_text().strip()) if time_file.exists() else 0.0
    done_epoch = checkpoint_epoch(latest)
    if done_epoch < epochs:
        args = [sys.executable, "tools/train.py", "-c", str(cfg_path)]
        if Path(str(latest) + ".pdparams").exists():
            args += ["-o", f"Global.checkpoints={latest}"]
        total_seconds += run_process(args, log_name)
        time_file.parent.mkdir(parents=True, exist_ok=True)
        time_file.write_text(str(total_seconds))
    return latest, total_seconds


def train_one(spec):
    name = spec["name"]
    cfg_path, save_dir = search_config_paths[name]
    latest, train_seconds = train_with_resume(
        cfg_path, save_dir, SEARCH_EPOCHS, f"{name}_train.log"
    )

    pred_txt = RESULTS_DIR / f"{name}_val.txt"
    infer_seconds = run_infer(
        cfg_path,
        VAL_FILE,
        pred_txt,
        f"{name}_val.log",
        checkpoint=latest,
    )
    metric, _ = evaluate_predictions(VAL_FILE, pred_txt)
    return {
        **spec,
        **metric,
        "train_seconds": train_seconds,
        "val_inference_seconds": infer_seconds,
        "config_path": str(cfg_path),
        "checkpoint": str(latest),
    }


search_results = []
for spec in CONFIGS:
    print("Running:", spec["name"])
    search_results.append(train_one(spec))

search_df = pd.DataFrame(search_results).sort_values(
    ["acc", "norm_edit_dis"], ascending=[False, False]
).reset_index(drop=True)
search_df.to_csv(RESULTS_DIR / "config_search_results.csv", index=False)
display(search_df)

### So sánh 640 và 960

In [ ]:
ablation_df = search_df[search_df["name"].isin(["c3_latin_640_aug", "c4_latin_960_aug"])][
    ["name", "width", "init", "recconaug", "acc", "norm_edit_dis", "train_seconds"]
].sort_values("width")
ablation_df.to_csv(RESULTS_DIR / "ablation_width.csv", index=False)
display(ablation_df)

## Chọn tốt nhất

In [ ]:
best_row = search_df.iloc[0]
best_spec = next(x for x in CONFIGS if x["name"] == best_row["name"])

with open(RESULTS_DIR / "best_config.json", "w", encoding="utf-8") as f:
    json.dump(best_spec, f, ensure_ascii=False, indent=2)

print("Best config:", best_spec)
print("Val acc:", best_row["acc"])
print("Val norm_edit_dis:", best_row["norm_edit_dis"])

## Fine-tune cuối

In [ ]:
FINAL_CONFIG, FINAL_SAVE_DIR = make_config(best_spec, FINAL_EPOCHS, "final")
FINAL_CKPT, final_train_seconds = train_with_resume(
    FINAL_CONFIG, FINAL_SAVE_DIR, FINAL_EPOCHS, "best_final_train.log"
)

FINAL_VAL_TXT = RESULTS_DIR / "best_final_val.txt"
run_infer(FINAL_CONFIG, VAL_FILE, FINAL_VAL_TXT, "best_final_val.log", checkpoint=FINAL_CKPT)
final_val_metric, _ = evaluate_predictions(VAL_FILE, FINAL_VAL_TXT)
print(final_val_metric)

## Test

In [ ]:
FINAL_TEST_TXT = RESULTS_DIR / "best_final_test.txt"
FINAL_TEST_JSONL = RESULTS_DIR / "pred_test_finetune.jsonl"

final_test_infer_seconds = run_infer(
    FINAL_CONFIG,
    TEST_FILE,
    FINAL_TEST_TXT,
    "best_final_test.log",
    checkpoint=FINAL_CKPT,
)
final_test_metric, final_test_records = evaluate_predictions(
    TEST_FILE,
    FINAL_TEST_TXT,
    FINAL_TEST_JSONL,
)

comparison = pd.DataFrame([
    {
        "model": "PaddleOCR goc",
        "acc": baseline_metric["acc"],
        "norm_edit_dis": baseline_metric["norm_edit_dis"],
        "train_seconds": 0.0,
    },
    {
        "model": "Fine-tune cua nhom",
        "acc": final_test_metric["acc"],
        "norm_edit_dis": final_test_metric["norm_edit_dis"],
        "train_seconds": final_train_seconds,
    },
])
comparison.to_csv(RESULTS_DIR / "comparison_test.csv", index=False)
display(comparison)

## Phân tích lỗi

Phân loại tự động chỉ là gợi ý; kiểm tra thủ công 20 dòng trước khi đưa vào báo cáo.

In [ ]:
TONE_MARKS = {"\u0300", "\u0301", "\u0303", "\u0309", "\u0323"}


def strip_tone(text):
    d = unicodedata.normalize("NFD", text)
    d = "".join(ch for ch in d if ch not in TONE_MARKS)
    return unicodedata.normalize("NFC", d)


def is_repeat_error(gt, pred):
    if len(pred) <= len(gt):
        return False
    for i in range(len(pred)):
        candidate = pred[:i] + pred[i + 1:]
        near_same = (i > 0 and pred[i] == pred[i - 1]) or (i + 1 < len(pred) and pred[i] == pred[i + 1])
        if candidate == gt and near_same:
            return True
    return False


def classify_error(gt, pred):
    if strip_tone(gt) == strip_tone(pred) and gt != pred:
        return "sai dau thanh"
    if is_repeat_error(gt, pred):
        return "lap ky tu"
    return "sai chu cai"

wrong = [r for r in final_test_records if metric_text(r["gt"]) != metric_text(r["pred"])]
rng = random.Random(SEED)
sample20 = rng.sample(wrong, min(20, len(wrong)))

for r in sample20:
    r["loai_loi_goi_y"] = classify_error(r["gt"], r["pred"])

error_df = pd.DataFrame(sample20)
error_df.to_csv(RESULTS_DIR / "error_analysis_20.csv", index=False)
display(error_df)

error_pct = (
    error_df["loai_loi_goi_y"]
    .value_counts(normalize=True)
    .mul(100)
    .rename_axis("loai_loi")
    .reset_index(name="phan_tram")
)
display(error_pct)

## Kết quả

In [ ]:
summary = {
    "seed": SEED,
    "paddle_version": paddle.__version__,
    "paddleocr_commit": PADDLEOCR_COMMIT,
    "max_text_length": MAX_TEXT_LENGTH,
    "search_epochs": SEARCH_EPOCHS,
    "final_epochs": FINAL_EPOCHS,
    "batch_size": BATCH_SIZE,
    "best_config": best_spec,
    "baseline": baseline_metric,
    "final_val": final_val_metric,
    "final_test": final_test_metric,
    "final_train_seconds": final_train_seconds,
    "final_test_inference_seconds": final_test_infer_seconds,
}
with open(RESULTS_DIR / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

bundle = shutil.make_archive(str(WORK / "paddleocr_vi_results"), "zip", root_dir=RESULTS_DIR)
print("Results:", RESULTS_DIR)
print("Bundle:", bundle)
print("Final checkpoint:", FINAL_CKPT)